In [1]:
from anytree import Node, RenderTree
import pandas as pd
import ast


In [9]:
def build_tree_recursive(parents, children):
    """
    parents  : list[Node]
    children : list[list[str]]  (one list per parent)
    """
    for parent, decay in zip(parents, children):
        for item in decay:
            if item == "-999":
                continue
            Node(item, parent=parent)

def print_anytree(root):
    for pre, fill, node in RenderTree(root):
        print(f"{pre}{node.name}")
        

def build_anytree_from_chain(chain, event_id=None):
    root = Node(f"Event {event_id}" if event_id else "Decay")

    # Start with root as the only parent
    parents = [root]

    for level in chain:
        # Remove -999
        level = [x for x in level if x != "-999"]

        new_parents = []

        # CASE 1: flat list → all come from all current parents
        if all(isinstance(x, str) for x in level):
            for parent in parents:
                for particle in level:
                    new_parents.append(Node(particle, parent=parent))

        # CASE 2: nested list → one sublist per parent
        else:
            for parent, decay in zip(parents, level):
                for particle in decay:
                    if particle != "-999":
                        new_parents.append(Node(particle, parent=parent))

        parents = new_parents

    return root


In [13]:
# Read CSV
csv_path = "/r02/lhcb/ejnw2/fcc_2025/FCCAnalyses/examples/FCCee/flavour/B2Inv/background_analysis/decay_analysis.csv"
df = pd.read_csv(csv_path)

# Parse the DecayChain column into real Python lists
df["DecayChain"] = df["DecayChain"].apply(ast.literal_eval)

In [17]:
event_id = 199

row = df.loc[df["Event"] == event_id].iloc[0]
chain = row["DecayChain"]

tree = build_anytree_from_chain(chain, event_id=event_id)
print_anytree(tree)

Event 199
└── b
    └── b
        ├── B^*-
        │   ├── B-
        │   │   ├── D^*(2007)0
        │   │   │   ├── D0
        │   │   │   │   ├── K-
        │   │   │   │   └── pi+
        │   │   │   └── gamma
        │   │   ├── Dbar0
        │   │   │   ├── K^*(892)0
        │   │   │   │   ├── K+
        │   │   │   │   └── pi-
        │   │   │   └── omega(782)0
        │   │   │       ├── pi-
        │   │   │       ├── pi+
        │   │   │       └── pi0
        │   │   │           ├── gamma
        │   │   │           └── gamma
        │   │   ├── pi0
        │   │   │   ├── gamma
        │   │   │   └── gamma
        │   │   └── K-
        │   └── gamma
        └── rho(770)+
            ├── pi+
            └── pi0
                ├── gamma
                └── gamma
